# 🎓 MEA Classroom Analytics - GPU Cloud Pipeline
This notebook executes the AI computer vision pipeline using Google Colab's free **T4 GPU**.

### Pipeline Flow:
1. **Mount Google Drive** to access input video & store output reports.
2. **Check GPU** to ensure CUDA acceleration.
3. **Clone Repo & Install Dependencies** (YOLOv8, OpenCV, ByteTrack).
4. **Configure Video File** from Drive (`MyDrive/MEA_Videos/`).
5. **Run Analysis** to generate student interaction scores & reports.
6. **View Results** and verify output saved to Drive.
7. *(Optional)* Launch ngrok tunnel to connect live with Electron frontend.

--- 
## 📁 Cell 1: Mount Google Drive
Mounts Google Drive to load your video and persist generated reports.

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Ensure the MEA_Videos directory exists
drive_videos_dir = '/content/drive/MyDrive/MEA_Videos'
os.makedirs(drive_videos_dir, exist_ok=True)
print(f"✓ Google Drive connected! Put your videos in: {drive_videos_dir}")


--- 
## ⚡ Cell 2: Verify GPU Acceleration (T4)
Checks that a GPU runtime is active.

In [ ]:
# Cell 2: Check GPU status
!nvidia-smi

import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"✓ Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Go to Runtime > Change runtime type > Select T4 GPU")


--- 
## 📦 Cell 3: Clone GitHub Repo & Install Dependencies
**Update the `GITHUB_REPO_URL` placeholder below** with your repository URL.

In [ ]:
# Cell 3: Clone repository & install dependencies
# =====================================================================
# >>> REPLACE THIS PLACEHOLDER WITH YOUR GITHUB REPO URL <<<
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
# =====================================================================

import os
repo_name = GITHUB_REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

# Clone if not already cloned in this Colab session
if not os.path.exists(f"/content/{repo_name}"):
    !git clone $GITHUB_REPO_URL /content/$repo_name
else:
    print(f"Repo already present at /content/{repo_name}. Pulling latest changes...")
    !cd /content/$repo_name && git pull

# Switch to cv-backend directory
if os.path.exists(f"/content/{repo_name}/cv-backend"):
    %cd /content/$repo_name/cv-backend
else:
    %cd /content/$repo_name

print("Current working directory:", os.getcwd())

# Install required dependencies
!pip install -q -r requirements.txt
!pip install -q pyngrok
print("✓ Dependencies installed successfully!")


--- 
## 🎥 Cell 4: Set Input Video & Output Paths
**Update the `VIDEO_FILENAME` placeholder below** with the actual file name of the video uploaded to `MyDrive/MEA_Videos/`.

In [ ]:
# Cell 4: Configure Video & Output Paths
# =====================================================================
# >>> REPLACE THIS PLACEHOLDER WITH YOUR ACTUAL VIDEO FILENAME <<<
VIDEO_FILENAME = "your_video.avi"
# =====================================================================

import os
from pathlib import Path

VIDEO_DIR = "/content/drive/MyDrive/MEA_Videos"
VIDEO_PATH = os.path.join(VIDEO_DIR, VIDEO_FILENAME)
base_name = Path(VIDEO_FILENAME).stem
OUTPUT_PATH = os.path.join(VIDEO_DIR, f"report_{base_name}.json")

if os.path.exists(VIDEO_PATH):
    file_size_mb = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
    print(f"✓ Video detected: {VIDEO_PATH} ({file_size_mb:.2f} MB)")
    print(f"✓ Output will be saved to: {OUTPUT_PATH}")
else:
    print(f"❌ ERROR: Video NOT found at: {VIDEO_PATH}")
    print("Please verify:")
    print(f"  1. Video is uploaded to: {VIDEO_DIR}")
    print(f"  2. VIDEO_FILENAME exactly matches (including .avi / .mp4)")
    if os.path.exists(VIDEO_DIR):
        available = os.listdir(VIDEO_DIR)
        print(f"Files currently in {VIDEO_DIR}: {available if available else '(Empty)'}")


--- 
## 🚀 Cell 5: Run Analytics Pipeline with T4 GPU
Executes person detection, tracking, proximity analysis, and isolation scoring.

In [ ]:
# Cell 5: Run the analytics pipeline
!python run.py \
    --video "$VIDEO_PATH" \
    --output "$OUTPUT_PATH" \
    --device 0 \
    --model yolov8n.pt

print("\n✓ Pipeline completed! Report saved.")


--- 
## 📊 Cell 6: Inspect & Summarize Report
Loads the saved `report.json` directly from Google Drive and prints the top metrics and least interactive students.

In [ ]:
# Cell 6: Inspect results
import json
import os

if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        report = json.load(f)

    print("=" * 60)
    print("             CLASSROOM ANALYTICS SUMMARY")
    print("=" * 60)
    print(f"Total Students Detected: {report.get('total_students', 'N/A')}")
    print(f"Video Duration: {report.get('duration_seconds', 0):.1f} s ({report.get('duration_seconds', 0)/60:.1f} min)")
    print(f"Frames Processed: {report.get('total_frames_processed', 'N/A')}")
    
    print("\n--- Top 3 Least Interactive Students ---")
    for idx, s in enumerate(report.get("top_3_least_interactive", []), 1):
        print(f"  {idx}. Student ID #{s.get('track_id')}")
        print(f"     • Interaction Score: {s.get('score', 0):.2f}")
        print(f"     • Isolated Time:     {s.get('isolated_minutes', 0):.1f} min")
        print(f"     • Near Peers Time:   {s.get('near_peer_minutes', 0):.1f} min")
        print(f"     • Confidence Score:  {s.get('confidence', 0):.2f}")
    print("=" * 60)
    print(f"✓ Full report permanently saved in your Google Drive at:")
    print(f"  {OUTPUT_PATH}")
    print("\nYou can now download this JSON and load it into your Electron Dashboard!")
else:
    print(f"❌ Report not found at {OUTPUT_PATH}. Please check Cell 5 output for any errors.")


--- 
## 🌐 (Optional) Cell 7: Live Backend Tunnel via ngrok for Electron Frontend
**Only needed if you want your local Electron app to connect directly to the running Colab GPU backend via an ngrok public tunnel URL.**
1. Sign up for a free ngrok account at https://dashboard.ngrok.com/get-started/your-authtoken
2. Paste your authtoken below and run this cell.

In [ ]:
# Optional Cell 7: Connect Electron to Colab via ngrok
# =====================================================================
# NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
# =====================================================================

# from pyngrok import ngrok
# ngrok.set_auth_token(NGROK_AUTHTOKEN)
# public_url = ngrok.connect(5000)
# print(f"\n✓ Ngrok Tunnel Created: {public_url}")
# print("Use this public URL in your Electron app API config!")
# !python api.py
